## import libraries

In [17]:
import numpy as np

##Problem setup

In [9]:
classes = [f'C{i+1}' for i in range(12)]
professors = {
    'C1': 'P1', 'C2': 'P1',
    'C3': 'P2', 'C4': 'P2',
    'C5': 'P3', 'C6': 'P3',
    'C7': 'P4', 'C8': 'P4',
    'C9': 'P5', 'C10': 'P5',
    'C11': 'P6', 'C12': 'P6'
}
timeslots = [f'T{i+1}' for i in range(6)]
n_classes = len(classes)
n_times = len(timeslots)
max_classes_per_time = 3


##Constraints

In [10]:
# Professors availability (each can only teach in some time slots)
availability = {
    'P1': [0, 1, 2],
    'P2': [1, 2, 3],
    'P3': [2, 3, 4],
    'P4': [0, 4, 5],
    'P5': [0, 2, 5],
    'P6': [1, 3, 5]
}

# Class pairs that cannot overlap (students overlap)
cannot_overlap = [('C1', 'C3'), ('C4', 'C6'), ('C2', 'C8'), ('C5', 'C10'), ('C7', 'C11')]


##Initialize pheromones and heuristic

In [11]:
pheromone = np.ones((n_classes, n_times))
heuristic = np.ones((n_classes, n_times))


##Ant builds a solution

In [12]:
#TODO: complete construct_solution function
def construct_solution(pheromone, heuristic, alpha=1, beta=2):
    schedule = []
    for i in range(n_classes):
        probs = (pheromone[i] ** alpha) * (heuristic[i] ** beta)
        probs /= probs.sum()
        time_index = np.random.choice(n_times, p=probs)
        schedule.append(time_index)
    return schedule


##Evaluate constraints

In [13]:
# TODO: Evaluate constraints
def evaluate_conflicts(schedule):
    conflicts = 0
    time_table = {t: [] for t in range(n_times)}

    for i, time in enumerate(schedule):
        cls = classes[i]
        prof = professors[cls]
        time_table[time].append((cls, prof))

    #Check that the same professor is not in multiple classes
    #Check the number of classes at any one time is not more than 3

    for time, assigned in time_table.items():
        profs = [p for c, p in assigned]
        if len(assigned) > max_classes_per_time:
            conflicts += (len(assigned) - max_classes_per_time)
        conflicts += len(profs) - len(set(profs))

    # Professor availability
    for i, time in enumerate(schedule):
        cls = classes[i]
        prof = professors[cls]
        if time not in availability[prof]:
            conflicts += 1

    # Forbidden class overlaps
    class_to_time = {cls: schedule[i] for i, cls in enumerate(classes)}
    for c1, c2 in cannot_overlap:
        if class_to_time[c1] == class_to_time[c2]:
            conflicts += 1

    return conflicts


##Pheromone update

In [14]:
#TODO: Pheromone update
def update_pheromone(pheromone, solutions, scores, rho=0.1, Q=100):
    pheromone *= (1 - rho)
    for sol, score in zip(solutions, scores):
        for i, t in enumerate(sol):
            pheromone[i][t] += Q / (1 + score)


##ACO main loop

In [15]:
#TODO: complete ACO main loop
n_ants = 10
n_iter = 30
best_sol = None
best_score = float('inf')
history = []

for it in range(n_iter):
    solutions = []
    scores = []
    for _ in range(n_ants):
        sol = construct_solution(pheromone, heuristic)
        score = evaluate_conflicts(sol)
        solutions.append(sol)
        scores.append(score)
        if score < best_score:
            best_score = score
            best_sol = sol
    update_pheromone(pheromone, solutions, scores)
    history.append(best_score)


##show result


In [16]:
print("Best schedule with minimum conflicts:")
for i, time in enumerate(best_sol):
    print(f"Class {classes[i]} → Time {timeslots[time]}")
print("Total conflicts:", best_score)


Best schedule with minimum conflicts:
Class C1 → Time T1
Class C2 → Time T3
Class C3 → Time T2
Class C4 → Time T4
Class C5 → Time T4
Class C6 → Time T5
Class C7 → Time T6
Class C8 → Time T1
Class C9 → Time T4
Class C10 → Time T2
Class C11 → Time T2
Class C12 → Time T3
Total conflicts: 3
